# Proyecto Hadoop: Resultados Presidenciales ONPE 2021 (Primera Vuelta)

Este notebook ejecuta un flujo **real** con Hadoop local (HDFS + MapReduce) usando el dataset oficial de ONPE: `presidencial-resultados-partidos.csv`.

## 1) Diagrama de diseño del sistema

```mermaid
flowchart LR
  onpeCsv[ONPECsvInput] --> normalizeStep[NormalizeColumnsStep]
  normalizeStep --> hdfsInput[HDFSInputPath]
  hdfsInput --> mapperNational[MapperCandidateVotes]
  hdfsInput --> mapperDept[MapperDeptCandidateVotes]
  mapperNational --> shuffleNational[ShuffleSortNational]
  mapperDept --> shuffleDept[ShuffleSortDepartment]
  shuffleNational --> reducerNational[ReducerSumVotes]
  shuffleDept --> reducerDept[ReducerSumVotes]
  reducerNational --> nationalResult[CandidateTotals]
  reducerDept --> deptResult[DepartmentCandidateTotals]
  nationalResult --> finalReport[WinnerAndPercentages]
```

## 2) Configuración mínima

In [3]:
import csv
import subprocess
from pathlib import Path

PROJECT_DIR = Path('.').resolve()
RAW_FILE = PROJECT_DIR / 'data' / 'raw_votes' / 'presidencial-resultados-partidos.csv'
NORMALIZED_FILE = PROJECT_DIR / 'data' / 'raw_votes' / 'onpe_normalized_votes.csv'
OUTPUT_DIR = PROJECT_DIR / 'output'

NAMENODE = 'hadoop-elections-namenode'
RESOURCEMANAGER = 'hadoop-elections-resourcemanager'
HDFS_INPUT = '/elections/input'
HDFS_OUTPUT_CANDIDATE = '/elections/output/candidate_totals'
HDFS_OUTPUT_DEPARTMENT = '/elections/output/department_candidate_totals'
STREAMING_JAR = '/opt/hadoop-3.2.1/share/hadoop/tools/lib/hadoop-streaming-3.2.1.jar'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Proyecto: {PROJECT_DIR}')
print(f'Dataset ONPE: {RAW_FILE}')

Proyecto: /Users/franspaxi/Downloads/data_engineering_course_lab2-main/hadoop_elections_real
Dataset ONPE: /Users/franspaxi/Downloads/data_engineering_course_lab2-main/hadoop_elections_real/data/raw_votes/presidencial-resultados-partidos.csv


In [4]:
def run(cmd, check=True):
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Fallo comando: {' '.join(cmd)}")
    return result

def docker_exec(container, command, check=True):
    return run(['docker', 'exec', container] + command, check=check)

def hdfs(args, check=True):
    return docker_exec(NAMENODE, ['hdfs', 'dfs'] + args, check=check)

def yarn_streaming(input_path, output_path, mapper, reducer):
    return docker_exec(
        RESOURCEMANAGER,
        [
            'yarn', 'jar', STREAMING_JAR,
            '-input', input_path,
            '-output', output_path,
            '-mapper', mapper,
            '-reducer', reducer,
            '-file', f'/mapreduce/{mapper}',
            '-file', f'/mapreduce/{reducer}',
        ],
    )

## 3) Prototipo local: normalizar dataset y subir a HDFS

Se crea un archivo simplificado con columnas:
- `departamento`
- `candidato`
- `total_votos`

Esto evita problemas por comas dentro del nombre del partido en el CSV original.

In [5]:
with RAW_FILE.open(newline='', encoding='utf-8') as src, NORMALIZED_FILE.open('w', newline='', encoding='utf-8') as dst:
    reader = csv.DictReader(src)
    writer = csv.writer(dst)
    writer.writerow(['departamento', 'candidato', 'total_votos'])
    rows = 0
    for row in reader:
        writer.writerow([row['departamento'], row['candidato'], row['total_votos']])
        rows += 1

print(f'Filas normalizadas: {rows}')
print(f'Archivo: {NORMALIZED_FILE}')

Filas normalizadas: 33732
Archivo: /Users/franspaxi/Downloads/data_engineering_course_lab2-main/hadoop_elections_real/data/raw_votes/onpe_normalized_votes.csv


In [6]:
run(['docker', 'compose', 'up', '-d'])
hdfs(['-rm', '-r', '-f', HDFS_INPUT, HDFS_OUTPUT_CANDIDATE, HDFS_OUTPUT_DEPARTMENT], check=False)
hdfs(['-mkdir', '-p', HDFS_INPUT])
hdfs(['-put', f'/data/raw_votes/{NORMALIZED_FILE.name}', f'{HDFS_INPUT}/{NORMALIZED_FILE.name}'])
hdfs(['-ls', HDFS_INPUT])

$ docker compose up -d
 Container hadoop-elections-namenode Running 
 Container hadoop-elections-datanode Running 
 Container hadoop-elections-resourcemanager Running 
 Container hadoop-elections-nodemanager Running 

$ docker exec hadoop-elections-namenode hdfs dfs -rm -r -f /elections/input /elections/output/candidate_totals /elections/output/department_candidate_totals
Deleted /elections/input
Deleted /elections/output/candidate_totals
Deleted /elections/output/department_candidate_totals

$ docker exec hadoop-elections-namenode hdfs dfs -mkdir -p /elections/input
$ docker exec hadoop-elections-namenode hdfs dfs -put /data/raw_votes/onpe_normalized_votes.csv /elections/input/onpe_normalized_votes.csv
2026-06-14 22:26:35,970 INFO sasl.SaslDataTransferClient: SASL encryption trust check: localHostTrusted = false, remoteHostTrusted = false

$ docker exec hadoop-elections-namenode hdfs dfs -ls /elections/input
Found 1 items
-rw-r--r--   3 root supergroup    1406658 2026-06-14 22:26 /ele

CompletedProcess(args=['docker', 'exec', 'hadoop-elections-namenode', 'hdfs', 'dfs', '-ls', '/elections/input'], returncode=0, stdout='Found 1 items\n-rw-r--r--   3 root supergroup    1406658 2026-06-14 22:26 /elections/input/onpe_normalized_votes.csv\n', stderr='')

## 4) Jobs MapReduce

In [7]:
# Job 1: votos totales por candidato
yarn_streaming(HDFS_INPUT, HDFS_OUTPUT_CANDIDATE, 'mapper_candidate.sh', 'reducer_count.sh')
hdfs(['-cat', f'{HDFS_OUTPUT_CANDIDATE}/part-*'])

$ docker exec hadoop-elections-resourcemanager yarn jar /opt/hadoop-3.2.1/share/hadoop/tools/lib/hadoop-streaming-3.2.1.jar -input /elections/input -output /elections/output/candidate_totals -mapper mapper_candidate.sh -reducer reducer_count.sh -file /mapreduce/mapper_candidate.sh -file /mapreduce/reducer_count.sh
packageJobJar: [/mapreduce/mapper_candidate.sh, /mapreduce/reducer_count.sh, /tmp/hadoop-unjar2028256809265765878/] [] /tmp/streamjob7533097888451010105.jar tmpDir=null

2026-06-14 22:26:39,796 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
2026-06-14 22:26:40,506 INFO client.RMProxy: Connecting to ResourceManager at resourcemanager/172.18.0.4:8032
2026-06-14 22:26:40,646 INFO client.AHSProxy: Connecting to Application History server at /0.0.0.0:10200
2026-06-14 22:26:40,677 INFO client.RMProxy: Connecting to ResourceManager at resourcemanager/172.18.0.4:8032
2026-06-14 22:26:40,677 INFO client.AHSProxy: Connecting to Applicati

CompletedProcess(args=['docker', 'exec', 'hadoop-elections-namenode', 'hdfs', 'dfs', '-cat', '/elections/output/candidate_totals/part-*'], returncode=0, stdout='ALBERTO ISMAEL BEINGOLEA DELGADO\t282007\nANDRES AVELINO ALCANTARA PAREDES\t50184\nCESAR ACUÑA PERALTA\t863955\nCIRO ALFREDO GALVEZ HERRERA\t88634\nDANIEL BELIZARIO URRESTI ELERA\t808559\nDANIEL ENRIQUE SALAVERRY VILLA\t237367\nGEORGE PATRICK FORSYTH SOMMER\t802957\nHERNANDO DE SOTO POLAR\t1652682\nJOSE ALEJANDRO VEGA ANTONIO\t100312\nJOSE PEDRO CASTILLO TERRONES\t2714152\nJULIO ARMANDO GUZMAN CACERES\t319166\nKEIKO SOFIA FUJIMORI HIGUCHI\t1907896\nMARCO ANTONIO ARANA ZEGARRA\t64217\nOLLANTA MOISES HUMALA TASSO\t228955\nRAFAEL BERNARDO LOPEZ ALIAGA CAZORLA\t1657575\nRAFAEL GASTON TADEO MILAGROS SANTOS NORMAND\t54341\nVERONIKA FANNY MENDOZA FRISCH\t1111407\nYONHY LESCANO ANCIETA\t1294681\n', stderr='2026-06-14 22:27:02,407 INFO sasl.SaslDataTransferClient: SASL encryption trust check: localHostTrusted = false, remoteHostTrusted 

In [8]:
# Job 2: votos por departamento y candidato
yarn_streaming(HDFS_INPUT, HDFS_OUTPUT_DEPARTMENT, 'mapper_city_candidate.sh', 'reducer_count.sh')
hdfs(['-cat', f'{HDFS_OUTPUT_DEPARTMENT}/part-*'])

$ docker exec hadoop-elections-resourcemanager yarn jar /opt/hadoop-3.2.1/share/hadoop/tools/lib/hadoop-streaming-3.2.1.jar -input /elections/input -output /elections/output/department_candidate_totals -mapper mapper_city_candidate.sh -reducer reducer_count.sh -file /mapreduce/mapper_city_candidate.sh -file /mapreduce/reducer_count.sh
packageJobJar: [/mapreduce/mapper_city_candidate.sh, /mapreduce/reducer_count.sh, /tmp/hadoop-unjar4663075772001819563/] [] /tmp/streamjob3243580684388638718.jar tmpDir=null

2026-06-14 22:27:04,053 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
2026-06-14 22:27:04,843 INFO client.RMProxy: Connecting to ResourceManager at resourcemanager/172.18.0.4:8032
2026-06-14 22:27:04,979 INFO client.AHSProxy: Connecting to Application History server at /0.0.0.0:10200
2026-06-14 22:27:05,008 INFO client.RMProxy: Connecting to ResourceManager at resourcemanager/172.18.0.4:8032
2026-06-14 22:27:05,009 INFO client.AHSProx

CompletedProcess(args=['docker', 'exec', 'hadoop-elections-namenode', 'hdfs', 'dfs', '-cat', '/elections/output/department_candidate_totals/part-*'], returncode=0, stdout='AMAZONAS|ALBERTO ISMAEL BEINGOLEA DELGADO\t492\nAMAZONAS|ANDRES AVELINO ALCANTARA PAREDES\t234\nAMAZONAS|CESAR ACUÑA PERALTA\t11531\nAMAZONAS|CIRO ALFREDO GALVEZ HERRERA\t774\nAMAZONAS|DANIEL BELIZARIO URRESTI ELERA\t4937\nAMAZONAS|DANIEL ENRIQUE SALAVERRY VILLA\t1926\nAMAZONAS|GEORGE PATRICK FORSYTH SOMMER\t4374\nAMAZONAS|HERNANDO DE SOTO POLAR\t4433\nAMAZONAS|JOSE ALEJANDRO VEGA ANTONIO\t2284\nAMAZONAS|JOSE PEDRO CASTILLO TERRONES\t34464\nAMAZONAS|JULIO ARMANDO GUZMAN CACERES\t1858\nAMAZONAS|KEIKO SOFIA FUJIMORI HIGUCHI\t17815\nAMAZONAS|MARCO ANTONIO ARANA ZEGARRA\t1285\nAMAZONAS|OLLANTA MOISES HUMALA TASSO\t15642\nAMAZONAS|RAFAEL BERNARDO LOPEZ ALIAGA CAZORLA\t8274\nAMAZONAS|RAFAEL GASTON TADEO MILAGROS SANTOS NORMAND\t279\nAMAZONAS|VERONIKA FANNY MENDOZA FRISCH\t8894\nAMAZONAS|YONHY LESCANO ANCIETA\t12703\nANCASH

## 5) Resultado final (reporte y archivos)

In [9]:
def read_hdfs_lines(path):
    out = hdfs(['-cat', path]).stdout.strip()
    return [line for line in out.splitlines() if line]

candidate_lines = read_hdfs_lines(f'{HDFS_OUTPUT_CANDIDATE}/part-*')
candidate_totals = []
for line in candidate_lines:
    candidate, votes = line.split('\t')
    candidate_totals.append((candidate, int(votes)))

candidate_totals.sort(key=lambda x: x[1], reverse=True)
grand_total = sum(v for _, v in candidate_totals)
winner = candidate_totals[0][0]

candidate_file = OUTPUT_DIR / 'candidate_totals.csv'
with candidate_file.open('w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['candidate', 'total_votes', 'percentage'])
    for cand, votes in candidate_totals:
        w.writerow([cand, votes, round((votes / grand_total) * 100, 2)])

dept_lines = read_hdfs_lines(f'{HDFS_OUTPUT_DEPARTMENT}/part-*')
dept_file = OUTPUT_DIR / 'department_candidate_totals.csv'
with dept_file.open('w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['departamento', 'candidate', 'total_votes'])
    for line in dept_lines:
        key, votes = line.split('\t')
        dept, cand = key.split('|', 1)
        w.writerow([dept, cand, int(votes)])

summary_file = OUTPUT_DIR / 'winner_summary.txt'
summary_text = (
    f'Ganador nacional: {winner}\n'
    f'Total votos válidos agregados: {grand_total}\n'
    'Top 5 candidatos:\n' + '\n'.join(
        f'  {c}: {v} ({round((v / grand_total) * 100, 2)}%)'
        for c, v in candidate_totals[:5]
    )
)
summary_file.write_text(summary_text, encoding='utf-8')

print('=== RESULTADO FINAL ===')
print(summary_text)
print(f'Archivos: {candidate_file.name}, {dept_file.name}, {summary_file.name}')

$ docker exec hadoop-elections-namenode hdfs dfs -cat /elections/output/candidate_totals/part-*
ALBERTO ISMAEL BEINGOLEA DELGADO	282007
ANDRES AVELINO ALCANTARA PAREDES	50184
CESAR ACUÑA PERALTA	863955
CIRO ALFREDO GALVEZ HERRERA	88634
DANIEL BELIZARIO URRESTI ELERA	808559
DANIEL ENRIQUE SALAVERRY VILLA	237367
GEORGE PATRICK FORSYTH SOMMER	802957
HERNANDO DE SOTO POLAR	1652682
JOSE ALEJANDRO VEGA ANTONIO	100312
JOSE PEDRO CASTILLO TERRONES	2714152
JULIO ARMANDO GUZMAN CACERES	319166
KEIKO SOFIA FUJIMORI HIGUCHI	1907896
MARCO ANTONIO ARANA ZEGARRA	64217
OLLANTA MOISES HUMALA TASSO	228955
RAFAEL BERNARDO LOPEZ ALIAGA CAZORLA	1657575
RAFAEL GASTON TADEO MILAGROS SANTOS NORMAND	54341
VERONIKA FANNY MENDOZA FRISCH	1111407
YONHY LESCANO ANCIETA	1294681

2026-06-14 22:27:30,413 INFO sasl.SaslDataTransferClient: SASL encryption trust check: localHostTrusted = false, remoteHostTrusted = false

$ docker exec hadoop-elections-namenode hdfs dfs -cat /elections/output/department_candidate_totals/pa

## 6) Preguntas de discusión

- **¿Por qué HDFS divide archivos en bloques?** Para distribuir almacenamiento/procesamiento y tolerar fallos con réplicas.
- **¿Responsabilidad del mapper?** Transformar cada fila de entrada en pares clave-valor (`candidato -> total_votos` o `departamento|candidato -> total_votos`).
- **¿Responsabilidad del reducer?** Agregar por clave y producir la suma final de votos.
- **¿Cuándo Hadoop vs script Python?** Hadoop es mejor con volumen y paralelismo distribuidos; Python simple basta para datasets pequeños en una sola máquina.